# Phase 2: Chunking & Embedding All Documents

This notebook:
1. Loads all Docling JSON documents
2. Chunks using HierarchicalChunker
3. Embeds with BGE-large
4. Stores in Milvus vector database
5. Tests retrieval queries
6. Generates answers using Gemini 2.0 Flash

## Cell 1: Imports & GPU Check

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
import json
from pathlib import Path
from pymilvus import MilvusClient
from tqdm import tqdm
from docling.chunking import HierarchicalChunker
from docling_core.types.doc import DoclingDocument
import time
from datetime import datetime
import google.generativeai as genai
import os

# Check CUDA availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: CUDA not available, using CPU")

## Cell 2: Load BGE Embedding Model

In [ ]:
# Load BGE embedding model
model_name = "BAAI/bge-large-en-v1.5"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {model_name} on {device}...")
embedding_model = SentenceTransformer(model_name, device=device)

# Test embedding to get dimension
test_embedding = embedding_model.encode(["test"])
embedding_dim = len(test_embedding[0])

print(f"\nModel loaded successfully!")
print(f"Model: {model_name}")
print(f"Device: {device}")
print(f"Embedding dimension: {embedding_dim}")
print(f"Max sequence length: {embedding_model.max_seq_length}")

## Cell 3: Configuration

In [ ]:
# Configuration
INPUT_DIR = Path("docling_outputs/json")
MILVUS_DB = "milvus_all_docs.db"
COLLECTION_NAME = "rag_all_docs"
BATCH_SIZE = 32
EMBEDDING_DIM = 1024

# Metadata fields
METADATA_FIELDS = ["source_file", "doc_type", "chunk_id", "total_chunks"]

print("Configuration:")
print(f"  Input directory: {INPUT_DIR}")
print(f"  Milvus DB: {MILVUS_DB}")
print(f"  Collection name: {COLLECTION_NAME}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Metadata fields: {METADATA_FIELDS}")

## Cell 4: Load All Docling JSON Documents

In [ ]:
# Scan for all JSON files
json_files = sorted(INPUT_DIR.glob("*.json"))

print(f"Found {len(json_files)} JSON files in {INPUT_DIR}\n")

# Load all documents
documents = {}
for json_file in json_files:
    with open(json_file, 'r', encoding='utf-8') as f:
        doc_data = json.load(f)
    documents[json_file.stem] = doc_data
    print(f"Loaded: {json_file.name}")

print(f"\nTotal documents loaded: {len(documents)}")
print(f"\nDocument files:")
for filename in documents.keys():
    print(f"  - {filename}")

## Cell 5: Chunk All Documents

In [ ]:
# Initialize HierarchicalChunker
chunker = HierarchicalChunker()

all_chunks = []
doc_type_counts = {}

print("Chunking all documents...\n")

for filename, doc_data in tqdm(documents.items(), desc="Chunking documents"):
    # Reconstruct Docling Document from JSON
    doc = DoclingDocument.model_validate(doc_data)
    
    # Get document type from origin or filename
    doc_type = "unknown"
    if "origin" in doc_data and "filename" in doc_data["origin"]:
        original_filename = doc_data["origin"]["filename"]
        doc_type = Path(original_filename).suffix.lstrip('.').lower() or "unknown"
    else:
        # Fallback to inferring from filename
        if "pdf" in filename.lower():
            doc_type = "pdf"
        elif "docx" in filename.lower():
            doc_type = "docx"
        elif "pptx" in filename.lower():
            doc_type = "pptx"
        elif "xlsx" in filename.lower():
            doc_type = "xlsx"
        elif "html" in filename.lower():
            doc_type = "html"
        elif "md" in filename.lower():
            doc_type = "md"
    
    # Apply chunker
    chunk_iter = chunker.chunk(doc)
    doc_chunks = list(chunk_iter)
    
    # Extract chunk texts and metadata
    for idx, chunk in enumerate(doc_chunks):
        chunk_text = chunk.text
        all_chunks.append({
            "text": chunk_text,
            "source_file": filename,
            "doc_type": doc_type,
            "chunk_id": idx,
            "total_chunks": len(doc_chunks)
        })
    
    # Track counts by doc type
    if doc_type not in doc_type_counts:
        doc_type_counts[doc_type] = 0
    doc_type_counts[doc_type] += len(doc_chunks)

print(f"\n{'='*60}")
print(f"Total chunks created: {len(all_chunks)}")
print(f"\nBreakdown by document type:")
for doc_type, count in sorted(doc_type_counts.items()):
    print(f"  {doc_type}: {count} chunks")
print(f"{'='*60}")

## Cell 6: Batch Embed All Chunks

In [ ]:
# Extract all chunk texts
all_texts = [chunk["text"] for chunk in all_chunks]

print(f"Embedding {len(all_texts)} chunks...")
start_time = time.time()

# Batch embed all texts
embeddings = embedding_model.encode(
    all_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True
)

# Convert to list of lists
embeddings_list = embeddings.tolist()

end_time = time.time()
elapsed_time = end_time - start_time

print(f"\n{'='*60}")
print(f"Embedding complete!")
print(f"Total chunks embedded: {len(embeddings_list)}")
print(f"Time taken: {elapsed_time:.2f} seconds")
print(f"Average time per chunk: {elapsed_time/len(embeddings_list)*1000:.2f} ms")
print(f"Embedding shape: {embeddings.shape}")
print(f"{'='*60}")

## Cell 7: Initialize Milvus & Create Collection

In [ ]:
# Initialize Milvus client
client = MilvusClient(MILVUS_DB)

# Drop collection if exists (fresh start)
if client.has_collection(COLLECTION_NAME):
    print(f"Dropping existing collection '{COLLECTION_NAME}'...")
    client.drop_collection(COLLECTION_NAME)

# Create collection with schema
print(f"Creating collection '{COLLECTION_NAME}'...")
client.create_collection(
    collection_name=COLLECTION_NAME,
    dimension=EMBEDDING_DIM,
    metric_type="IP",  # Inner Product
    consistency_level="Strong"
)

print(f"\nCollection '{COLLECTION_NAME}' created successfully!")
print(f"  Dimension: {EMBEDDING_DIM}")
print(f"  Metric type: IP (Inner Product)")
print(f"  Consistency level: Strong")
print(f"  Metadata fields: {METADATA_FIELDS}")

## Cell 8: Insert Data into Milvus

In [ ]:
# Prepare data for insertion
data_to_insert = []

for idx, (chunk, embedding) in enumerate(zip(all_chunks, embeddings_list)):
    data_to_insert.append({
        "id": idx,
        "vector": embedding,
        "text": chunk["text"],
        "source_file": chunk["source_file"],
        "doc_type": chunk["doc_type"],
        "chunk_id": chunk["chunk_id"],
        "total_chunks": chunk["total_chunks"]
    })

print(f"Inserting {len(data_to_insert)} records into Milvus...")
start_time = time.time()

# Insert all at once
insert_result = client.insert(
    collection_name=COLLECTION_NAME,
    data=data_to_insert
)

end_time = time.time()

print(f"\n{'='*60}")
print(f"Insertion complete!")
print(f"Records inserted: {insert_result['insert_count']}")
print(f"Time taken: {end_time - start_time:.2f} seconds")
print(f"{'='*60}")

## Cell 9: Test Search - Financial Statements Query

In [ ]:
# Query 1: Financial statements
query_text = "What are the financial statements prepared according to Indian Accounting Standards for FY 23-24?"

print(f"Query: {query_text}\n")
print("Embedding query...")
query_embedding = embedding_model.encode([query_text])[0].tolist()

print("Searching Milvus...")
search_results = client.search(
    collection_name=COLLECTION_NAME,
    data=[query_embedding],
    limit=10,
    output_fields=["text", "source_file", "doc_type", "chunk_id", "total_chunks"]
)

print(f"\n{'='*80}")
print(f"Top 10 Results:")
print(f"{'='*80}\n")

for idx, result in enumerate(search_results[0], 1):
    print(f"Result {idx}:")
    print(f"  Source: {result['entity']['source_file']}")
    print(f"  Doc Type: {result['entity']['doc_type']}")
    print(f"  Chunk: {result['entity']['chunk_id']+1}/{result['entity']['total_chunks']}")
    print(f"  Distance: {result['distance']:.4f}")
    print(f"  Text: {result['entity']['text'][:200]}...")
    print()

## Cell 10: Test Search - Balance Sheet Query

In [ ]:
# Query 2: Balance sheet
query_text = "Show me balance sheet information for ExampleCo"

print(f"Query: {query_text}\n")
print("Embedding query...")
query_embedding = embedding_model.encode([query_text])[0].tolist()

print("Searching Milvus...")
search_results = client.search(
    collection_name=COLLECTION_NAME,
    data=[query_embedding],
    limit=10,
    output_fields=["text", "source_file", "doc_type", "chunk_id", "total_chunks"]
)

print(f"\n{'='*80}")
print(f"Top 10 Results:")
print(f"{'='*80}\n")

for idx, result in enumerate(search_results[0], 1):
    print(f"Result {idx}:")
    print(f"  Source: {result['entity']['source_file']}")
    print(f"  Doc Type: {result['entity']['doc_type']}")
    print(f"  Chunk: {result['entity']['chunk_id']+1}/{result['entity']['total_chunks']}")
    print(f"  Distance: {result['distance']:.4f}")
    print(f"  Text: {result['entity']['text'][:200]}...")
    print()

## Cell 11: Collection Statistics & Summary

In [ ]:
# Get collection statistics
stats = client.get_collection_stats(COLLECTION_NAME)

# Calculate breakdown by doc_type
doc_type_breakdown = {}
for chunk in all_chunks:
    doc_type = chunk["doc_type"]
    if doc_type not in doc_type_breakdown:
        doc_type_breakdown[doc_type] = 0
    doc_type_breakdown[doc_type] += 1

# Calculate per-document breakdown
per_doc_breakdown = {}
for chunk in all_chunks:
    source = chunk["source_file"]
    if source not in per_doc_breakdown:
        per_doc_breakdown[source] = 0
    per_doc_breakdown[source] += 1

print(f"\n{'='*80}")
print(f"COLLECTION STATISTICS")
print(f"{'='*80}")
print(f"\nCollection Name: {COLLECTION_NAME}")
print(f"Total Vectors: {stats['row_count']}")
print(f"Embedding Dimension: {EMBEDDING_DIM}")

print(f"\nBreakdown by Document Type:")
for doc_type, count in sorted(doc_type_breakdown.items()):
    percentage = (count / len(all_chunks)) * 100
    print(f"  {doc_type}: {count} chunks ({percentage:.1f}%)")

print(f"\nPer-Document Chunk Counts:")
for source, count in sorted(per_doc_breakdown.items()):
    print(f"  {source}: {count} chunks")

avg_chunks = len(all_chunks) / len(documents)
print(f"\nAverage chunks per document: {avg_chunks:.1f}")
print(f"Total documents processed: {len(documents)}")
print(f"Total chunks created: {len(all_chunks)}")

# Estimate collection size
vector_size = len(all_chunks) * EMBEDDING_DIM * 4 / (1024 * 1024)  # 4 bytes per float32, convert to MB
print(f"\nEstimated collection size: {vector_size:.2f} MB (vectors only)")
print(f"{'='*80}")

## Cell 12: Save Chunking Report

In [ ]:
# Create chunking report
report = {
    "processing_info": {
        "timestamp": datetime.now().isoformat(),
        "total_documents_processed": len(documents),
        "total_chunks_created": len(all_chunks),
        "embedding_model": model_name,
        "milvus_collection": COLLECTION_NAME,
        "milvus_db_file": MILVUS_DB
    },
    "chunking_statistics": {
        "average_chunks_per_document": len(all_chunks) / len(documents),
        "chunks_by_doc_type": doc_type_breakdown,
        "chunks_by_document": per_doc_breakdown
    },
    "embedding_info": {
        "model_name": model_name,
        "dimension": EMBEDDING_DIM,
        "batch_size": BATCH_SIZE,
        "device": device
    },
    "milvus_info": {
        "collection_name": COLLECTION_NAME,
        "metric_type": "IP",
        "total_vectors": stats['row_count'],
        "metadata_fields": METADATA_FIELDS
    },
    "document_list": list(documents.keys())
}

# Save report
report_path = Path("docling_outputs/chunking_report.json")
report_path.parent.mkdir(parents=True, exist_ok=True)

with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(f"Chunking report saved to: {report_path}")
print(f"\nReport summary:")
print(json.dumps(report, indent=2))

## Cell 13: Initialize Gemini 2.0 Flash for Answer Generation

In [ ]:
# Configure Gemini API
# Note: Set your GEMINI_API_KEY or GOOGLE_API_KEY environment variable
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

if not GEMINI_API_KEY:
    print("WARNING: GEMINI_API_KEY not found in environment variables")
    print("Please set GEMINI_API_KEY or GOOGLE_API_KEY to use Gemini for answer generation")
    print("Example: export GEMINI_API_KEY='your-api-key-here'")
else:
    genai.configure(api_key=GEMINI_API_KEY)
    gemini_model = genai.GenerativeModel('gemini-2.5-flash-lite')
    
    print("Gemini 2.0 Flash initialized successfully!")
    print(f"Model: gemini-2.5-flash-lite")
    print(f"Ready for RAG answer generation")

## Cell 14: RAG Function - Retrieve Context & Generate Answer

In [ ]:
def rag_query(query_text, top_k=5, show_full_chunks=True):
    """
    Complete RAG pipeline: Retrieve relevant chunks and generate answer using Gemini.
    
    Args:
        query_text: The question to answer
        top_k: Number of chunks to retrieve
        show_full_chunks: Whether to print full chunk text
    
    Returns:
        dict with answer, sources, and metadata
    """
    print(f"\n{'='*80}")
    print(f"QUERY: {query_text}")
    print(f"{'='*80}\n")
    
    # Step 1: Embed query
    print(f"Step 1: Embedding query...")
    query_embedding = embedding_model.encode([query_text])[0].tolist()
    
    # Step 2: Search vector database
    print(f"Step 2: Searching for top {top_k} relevant chunks...")
    search_results = client.search(
        collection_name=COLLECTION_NAME,
        data=[query_embedding],
        limit=top_k,
        output_fields=["text", "source_file", "doc_type", "chunk_id", "total_chunks"]
    )
    
    # Extract results
    context_texts = []
    sources = []
    
    print(f"\n{'='*80}")
    print(f"RETRIEVED CHUNKS:")
    print(f"{'='*80}\n")
    
    for idx, result in enumerate(search_results[0], 1):
        entity = result['entity']
        chunk_text = entity['text']
        context_texts.append(chunk_text)
        
        source_info = {
            'rank': idx,
            'source_file': entity['source_file'],
            'doc_type': entity['doc_type'],
            'chunk_id': entity['chunk_id'],
            'total_chunks': entity['total_chunks'],
            'distance': result['distance'],
            'text': chunk_text
        }
        sources.append(source_info)
        
        # Display chunk info
        print(f"Chunk {idx}:")
        print(f"  Source: {entity['source_file']} ({entity['doc_type']})")
        print(f"  Chunk: {entity['chunk_id']+1}/{entity['total_chunks']}")
        print(f"  Distance: {result['distance']:.4f}")
        if show_full_chunks:
            print(f"  Text:\n{chunk_text}")
        else:
            print(f"  Text: {chunk_text[:200]}...")
        print(f"\n{'-'*80}\n")
    
    # Step 3: Build context for Gemini
    print(f"Step 3: Building context for LLM...")
    context = "\n\n".join([
        f"[Source {i+1}: {sources[i]['source_file']}]\n{text}" 
        for i, text in enumerate(context_texts)
    ])
    
    # Step 4: Generate answer with Gemini
    print(f"Step 4: Generating answer with Gemini 2.5 Flash...\n")
    
    prompt = f"""You are a helpful AI assistant that answers questions based on provided document context.

Context from documents:
{context}

Question: {query_text}

Please provide a comprehensive answer based on the context above. 
- If the context contains relevant information, synthesize it into a clear answer.
- If the context doesn't contain enough information, explicitly state that.
- Cite which sources you're using when answering."""
    
    try:
        response = gemini_model.generate_content(prompt)
        answer = response.text
    except Exception as e:
        answer = f"Error generating answer: {str(e)}"
    
    # Display answer
    print(f"{'='*80}")
    print(f"GENERATED ANSWER:")
    print(f"{'='*80}")
    print(answer)
    print(f"\n{'='*80}\n")
    
    return {
        'question': query_text,
        'answer': answer,
        'sources': sources,
        'num_chunks_retrieved': len(context_texts)
    }

print("RAG function ready!")
print("Usage: rag_query('your question here', top_k=5, show_full_chunks=True)")

## Cell 15: Test RAG - Financial Statements Query

In [ ]:
# Test RAG with financial statements query
result1 = rag_query(
    "What are the financial statements prepared according to Indian Accounting Standards for FY 23-24?",
    top_k=5,
    show_full_chunks=True
)

## Cell 16: Test RAG - Balance Sheet Query

In [ ]:
# Test RAG with balance sheet query
result2 = rag_query(
    "Show me balance sheet information for ExampleCo",
    top_k=5,
    show_full_chunks=True
)

## Cell 17: Custom Query Interface

In [ ]:
# Use this cell to run your own custom queries
# Uncomment and modify the query below:
questions = [
            #Page 1 questions
            "Language (English/Hindi)"
            "Authorised capital of the company as on the date of filling (in Rs.)",
            "1.(a) Corporate Identity Number (CIN) of company",
            "2.(a) Name of the company",
            "2.(b) Address of the registered office of the company",
            "2.(c) e-mail ID of the company",
            "3. Financial year to which financial statement relates - From:____ (DD/MM/YYYY) To:____ (DD/MM/YYYY)",
            "4.(a) Date of Board of Director's meeting in which financial statements are approved - ____ (DD/MM/YYYY)",
            "4.(b) Nature of financial statements: ____? 4.(c) Whether provisional financial statements filed earlier (Yes/No/Not appicable). 4.(d) Whether adopted in adjourned AGM (Yes/No/Not appicable)",
            "5.(a) Whether Annual General Meeting (AGM) held (Yes/No/Not appicable), 5.(b) if YES then date of AGM:____ (DD/MM/YYYY). 5.(c) Due date of AGM:____ (DD/MM/YYYY). 5.(d) Whether any extension for financial year or AGM granted (Yes/No)",
            "6.(a) Whether Schedule 3 of the Companies Act, 2013 is applicable (Yes/No). 6.(b) Whether financial statements have been drawn on the basis of ____ (AS Taxonomy/Ind-AS Taxonomy)",

            #Page 2 questions
            "7. Type of Industry",
            "8. Whether consolidated Financial State,emts are also being filed (Yes/No)",
            "9.(a) In case if a government company, whether CAG of India has commented upon of supplemented the audit report under section 143 of the Companies Act, 2013 (Yes/No)",
            "9.(d) whether CAG of India has conducted supplementary or test audit under section 143 (Yes/No)",
            "10. Whether Secretarial Audit is applicable (Yes/No)"
            "11. Whether detailed discloure with respect to Director's report Sec 134(3) is attached (Yes/No)"
    ]
for question in questions:
    custom_result = rag_query(
        question,
        top_k=5,
        show_full_chunks=True
    )


In [ ]:
custom_result = rag_query(
        "Address of the company",
        top_k=5,
        show_full_chunks=True
    )

## Summary

This notebook has:
1. Loaded all Docling JSON documents from `docling_outputs/json/`
2. Chunked them using HierarchicalChunker
3. Embedded all chunks using BGE-large (1024 dimensions)
4. Stored embeddings in Milvus vector database
5. Implemented semantic search across all documents
6. Integrated Gemini 2.0 Flash for answer generation
7. Created a complete RAG pipeline for question answering

### Key Features:
- **Single collection** for all document types with metadata
- **Batch embedding** for maximum performance
- **Rich metadata** including source file, doc type, and chunk IDs
- **Full RAG pipeline** with context retrieval and LLM generation
- **Reusable function** for custom queries

### Next Steps:
- Use the `rag_query()` function for any questions about your documents
- Experiment with different `top_k` values for retrieval
- Add metadata filtering for specific document types
- Export results for downstream applications